In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
tmdb = pd.read_csv("../datasets/raw/tmdb-movies/TMDB_movie_dataset_v11.csv")
KEEP = [
    "id",
    "title",
    "overview",
    "genres",
    "keywords",
    "vote_average",
    "vote_count",
    "popularity",
    "release_date"
]
tmdb = tmdb[KEEP]
tmdb = tmdb[tmdb["vote_count"] > 30]
tmdb["overview"] = tmdb["overview"].fillna("")
tmdb["genres"] = tmdb["genres"].fillna("")
tmdb["keywords"] = tmdb["keywords"].fillna("")
tmdb.head()

In [ ]:
plt.hist(tmdb["popularity"], bins=50)
plt.title("Raw popularity")
plt.show()

plt.hist(np.log1p(tmdb["popularity"]), bins=50)
plt.title("Log-transformed popularity")
plt.show()

In [ ]:
tmdb["popularity_log"] = np.log1p(tmdb["popularity"])
tmdb["content"] = tmdb["overview"] + " " + tmdb["genres"] + " " + tmdb["keywords"]

In [ ]:
tmdb.info()

In [ ]:
movie_lens_links = pd.read_csv("../datasets/raw/ml-32m/links.csv")

movie_lens_links = movie_lens_links.dropna(subset=["tmdbId"])
movie_lens_links["tmdbId"] = movie_lens_links["tmdbId"].astype("int64")
movie_lens_links.head()

In [ ]:
movie_lens_ratings = pd.read_csv("../datasets/raw/ml-32m/ratings.csv")
ratings = pd.merge(
    movie_lens_ratings[['userId', 'rating', 'movieId']],
    movie_lens_links[['movieId', 'tmdbId']],
    on='movieId',
    how='left'
)
ratings = ratings[ratings["tmdbId"].notna()]
ratings["tmdbId"] = ratings["tmdbId"].astype("int64")

min_user_ratings = 5
min_movie_ratings = 5

user_counts = ratings.groupby('userId').size()
movie_counts = ratings.groupby('tmdbId').size()

rating = ratings[
    ratings['userId'].isin(user_counts[user_counts >= min_user_ratings].index) &
    ratings['tmdbId'].isin(movie_counts[movie_counts >= min_movie_ratings].index)
]
ratings.head()

In [ ]:
ratings.info()